# [PBT] 화장품 이커머스 ㅣ 09. 단변량 — **A·B 통합**

## 이 노트북이 하는 일

A조·B조가 각자 만든 CSV를 합쳐 **9개 변수 한 표**를 만든다.

교재가 단변량의 마지막에 요구하는 것이 이 표다 — `[LAB-08 2단원 p.22]`
> *"다음과 같이 표로 분석 결과를 요약해서 **모델링 설계의 근거**로 삼는다."*

이 표가 두 곳으로 간다.

| 어디로 | 무엇으로 |
|---|---|
| **보고서 Ⅱ-1 단변량** | 표 그대로 |
| **⑤ 모델링용 전처리** | 로그변환 대상 목록 |

## 입력

| 파일 | 만든 사람 | 노트북 |
|---|---|---|
| `output/09a_단변량_A조.csv` | 이슬기 | `09_단변량_A조_조회변수` |
| `output/09b_단변량_B조.csv` | 배지환 | `09_단변량_B조_상품상황_v2` |

> **두 CSV가 모두 최신인지 먼저 확인한다.** 각 노트북을 **Run All** 한 뒤에 이 노트북을 돌린다.
> 특히 B조 v2는 미결 ① 결정(`log1p` 통일)이 반영된 판이어야 한다.

## 여기서 하지 않는 것

| 안 하는 것 | 어디서 |
|---|---|
| 로그변환 **실행** | ⑤ 전처리, **층화 분할 후** |
| Y와의 관계 | `10` 이변량 |
| 변수 선택 | `10` 다변량 |

---

## #01. 준비

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)
sys.path.insert(0, BASE)

from pandas import read_csv, concat, set_option, isna
set_option("display.max_columns", 50)
set_option("display.max_colwidth", 60)      # 판단·조치가 길어 잘리는 것을 막는다

OUT = BASE + r"\output"

A_PATH = OUT + r"\09a_단변량_A조.csv"
B_PATH = OUT + r"\09b_단변량_B조.csv"

for p in [A_PATH, B_PATH]:
    print(("있음  " if os.path.exists(p) else "★ 없음") , p)

있음   .\output\09a_단변량_A조.csv
있음   .\output\09b_단변량_B조.csv


### 1-1. 두 파일을 읽는다

`index_col=0` — 저장할 때 변수명이 인덱스로 들어갔다.

In [2]:
A = read_csv(A_PATH, index_col=0, encoding="utf-8-sig")
B = read_csv(B_PATH, index_col=0, encoding="utf-8-sig")

print(f"A조 : {A.shape[0]}행 × {A.shape[1]}열   {list(A.columns)}")
print()
print(f"B조 : {B.shape[0]}행 × {B.shape[1]}열   {list(B.columns)}")

A조 : 4행 × 11열   ['담당', '척도', 'mean', '50%', 'max', 'skew', '0의 비율%', '이상치%', '고유값 수', '로그변환 후보', '판단·조치']

B조 : 5행 × 11열   ['담당', '척도', '범주 수/고유값', '왜도', '첨도', '이상치%', '로그변환 후보', '0의 비율%', '최빈 비율%', '최소 비율%', '판단·조치']


---

## #02. 합치기 전 검증

**두 표의 컬럼이 다르다.** A조는 연속형 중심(`mean`·`0의 비율%`), B조는 명목형 중심(`최빈 비율%`)이다.
그대로 이어 붙이면 컬럼이 뒤엉키므로 **공통 이름으로 맞춘 뒤** 합친다.

| 공통 이름 | A조에서 | B조에서 |
|---|---|---|
| `종수/고유값` | `고유값 수` | `범주 수/고유값` |
| `왜도` | `skew` | `왜도` |
| 나머지 | 있으면 가져오고 없으면 비운다 | 〃 |

In [3]:
# ── 공통 스키마로 이름 맞추기 ────────────────────────────────────
A2 = A.rename(columns={"고유값 수": "종수/고유값", "skew": "왜도"})
B2 = B.rename(columns={"범주 수/고유값": "종수/고유값"})

COLS = ["담당", "척도", "종수/고유값", "왜도", "첨도",
        "0의 비율%", "이상치%", "최빈 비율%", "최소 비율%",
        "로그변환 후보", "판단·조치"]

# 없는 컬럼은 만들어 비워 둔다 (A조엔 최빈/최소, B조엔 0의 비율이 없다)
for d in (A2, B2):
    for c in COLS:
        if c not in d.columns:
            d[c] = None

U = concat([A2[COLS], B2[COLS]])
U.index.name = "변수"

# ── 이진 변수의 최빈/최소 비율 채우기 ────────────────────────────
# A조 표에는 '0의 비율%' 만 있다. 이진이면 두 범주뿐이므로
# 최소 = 0의 비율, 최빈 = 100 - 0의 비율 로 유도할 수 있다 (새 정보가 아니다).
for v in U.index[U["척도"] == "이진"]:
    z = float(U.loc[v, "0의 비율%"])
    U.loc[v, "최소 비율%"] = round(min(z, 100 - z), 2)
    U.loc[v, "최빈 비율%"] = round(max(z, 100 - z), 2)

print(f"합친 결과 : {U.shape[0]}행 × {U.shape[1]}열")
print("이진 변수의 최빈/최소 비율은 '0의 비율%' 에서 유도했다 (계산일 뿐 새 정보가 아니다).")

합친 결과 : 9행 × 11열
이진 변수의 최빈/최소 비율은 '0의 비율%' 에서 유도했다 (계산일 뿐 새 정보가 아니다).


### 2-1. 검증 5종 — **하나라도 실패하면 멈춘다**

합치기는 조용히 실패하기 쉽다. 행이 빠져도, 이름이 어긋나도 표는 그냥 만들어진다.

In [4]:
# 브리핑 §4 변수 역할표 : 모델 X 8개 + 대조용 category_grp = 9개
EXPECT = ["prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt", "has_prior_view",
          "price", "brand_grp", "cart_timeband", "cart_weekday", "category_grp"]

chk = []
chk.append(("행 수 9개",           U.shape[0] == 9,                      f"{U.shape[0]}행"))
chk.append(("변수명 중복 없음",     not U.index.duplicated().any(),       f"중복 {int(U.index.duplicated().sum())}건"))
chk.append(("기대 변수와 일치",     set(U.index) == set(EXPECT),
            f"빠짐 {sorted(set(EXPECT) - set(U.index))} / 여분 {sorted(set(U.index) - set(EXPECT))}"))
chk.append(("판단·조치 공란 없음",  U["판단·조치"].notna().all() and (U["판단·조치"].astype(str).str.strip() != "").all(),
            f"공란 {int((U['판단·조치'].astype(str).str.strip() == '').sum())}건"))
chk.append(("로그변환 후보 4개",    int(U["로그변환 후보"].sum()) == 4,   f"{int(U['로그변환 후보'].sum())}개"))

from pandas import DataFrame
V = DataFrame(chk, columns=["검증", "통과", "값"])
V

,검증,통과,값
0,행 수 9개,True,9행
1,변수명 중복 없음,True,중복 0건
2,기대 변수와 일치,True,빠짐 [] / 여분 []
3,판단·조치 공란 없음,True,공란 0건
4,로그변환 후보 4개,True,4개


> **`로그변환 후보`가 4개여야 하는 이유** — 연속형 변수가 `price` + 조회 3종으로 **정확히 4개**다.
> 명목형·이진은 애초에 대상이 아니다. 5개나 3개가 나오면 어느 한쪽 CSV가 옛 판이다.

> **`판단·조치 공란 없음`** — 실제로 한 번 사고가 났던 항목이다.
> 노트북에는 판정을 써두고 CSV에는 옮기지 않아 빈 칸으로 전달된 적이 있다.

---

## #03. 통합표

교재가 요구한 **"모델링 설계의 근거"** 표다 `[LAB-08 2단원 p.22]`.

In [5]:
# 보기 좋은 순서로 — 조회 행동 → 상품·상황 → 기록용
ORDER = ["prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt", "has_prior_view",
         "price", "brand_grp", "cart_timeband", "cart_weekday", "category_grp"]

U = U.loc[ORDER]
U.round(3)

,담당,척도,종수/고유값,왜도,첨도,0의 비율%,이상치%,최빈 비율%,최소 비율%,로그변환 후보,판단·조치
변수,,,,,,,,,,,
prior_view_cnt,A · 이슬기,연속,51,5.901,None,44.68,13.79,None,None,True,"로그변환 대상(log1p, 5.901→1.042) · 변환 후에도 기준 1.0 초과 → 미결 ① · ..."
prior_uniq_product_cnt,A · 이슬기,연속,46,6.044,None,44.68,11.02,None,None,True,"로그변환 대상(log1p, 6.044→1.033) · 변환 후에도 기준 1.0 초과 → 미결 ① · ..."
cart_product_view_cnt,A · 이슬기,연속,10,1.751,None,61.30,1.14,None,None,True,"로그변환 대상(log1p, 1.751→0.809, 셋 중 유일하게 기준 통과) · 연속형 유지 · 고..."
has_prior_view,A · 이슬기,이진,2,NaN,None,44.68,NaN,55.32,44.68,False,그대로 사용 가능 · 55.32% : 44.68% 로 균형 · 검정력 문제 없음
price,B · 배지환,연속,1708,7.210,69.03,0.00,9.00,NaN,NaN,True,"로그변환 대상(log1p, 왜도 7.21→0.81) · 네 변수 log1p 통일(미결 ① 결정) · ..."
brand_grp,B · 배지환,명목,15,NaN,NaN,NaN,NaN,38.85,1.42,False,그대로 사용 · 소수범주 유지 · '미상'+'기타' 62.48%라 두 범주 계수는 브랜드 효과로 해석...
cart_timeband,B · 배지환,명목,4,NaN,NaN,NaN,NaN,36.13,8.4,False,"그대로 사용 가능 · 최소 범주도 9,107명이라 판정불가 구간 없음 · 순서는 정렬에만 쓰이고 모형..."
cart_weekday,B · 배지환,명목,7,NaN,NaN,NaN,NaN,16.81,12.89,False,그대로 사용 · 분포 서술은 반드시 일평균 기준(보정 시 1위가 금→목) · 보정 후 주중>주말 패턴...
category_grp,B · 배지환,명목(기록용),59,NaN,NaN,NaN,NaN,32.71,0.46,False,판정하지 않음 · 06번에서 X 제외 확정 · '기타' 32.71%는 한계 문장의 근거 · 결정12 ...


### 3-1. 척도별로 나눠 보기

한 표에 연속형과 명목형을 같이 두면 빈 칸이 많아 읽기 어렵다. **보고서에는 나눠 싣는다.**

In [6]:
CONT = U[U["척도"].isin(["연속"])]
NOMI = U[~U["척도"].isin(["연속"])]

print("── 연속형 4개 : 분포·이상치·변환 판정 ──")
display(CONT[["담당", "종수/고유값", "왜도", "0의 비율%", "이상치%", "로그변환 후보"]].round(3))

print()
print("── 이진·명목형 5개 : 범주 균형 ──")
display(NOMI[["담당", "척도", "종수/고유값", "최빈 비율%", "최소 비율%"]].round(2))

── 연속형 4개 : 분포·이상치·변환 판정 ──


,담당,종수/고유값,왜도,0의 비율%,이상치%,로그변환 후보
변수,,,,,,
prior_view_cnt,A · 이슬기,51,5.901,44.68,13.79,True
prior_uniq_product_cnt,A · 이슬기,46,6.044,44.68,11.02,True
cart_product_view_cnt,A · 이슬기,10,1.751,61.30,1.14,True
price,B · 배지환,1708,7.210,0.00,9.00,True



── 이진·명목형 5개 : 범주 균형 ──


,담당,척도,종수/고유값,최빈 비율%,최소 비율%
변수,,,,,
has_prior_view,A · 이슬기,이진,2,55.32,44.68
brand_grp,B · 배지환,명목,15,38.85,1.42
cart_timeband,B · 배지환,명목,4,36.13,8.4
cart_weekday,B · 배지환,명목,7,16.81,12.89
category_grp,B · 배지환,명목(기록용),59,32.71,0.46


---

## #04. ⑤ 전처리로 넘길 것

단변량의 결론 중 **다음 단계가 실제로 실행해야 하는 것**만 뽑는다.

In [7]:
todo = U[U["로그변환 후보"] == True]

print("⑤ 모델링용 전처리에서 log1p 변환할 변수")
print()
for v, r in todo.iterrows():
    print(f"  - {v:<24} 왜도 {float(r['왜도']):>6.3f}  (담당 {r['담당']})")

print()
print("※ 변환은 log 가 아니라 log1p 로 통일한다 — 미결 ① 결정 (2026-08-31, 임시).")
print("※ 반드시 '층화 분할 후'에 변환한다 (LAB-13 04).")
print("※ 이상치는 처리하지 않는다 (설계 §3-5).")
print("※ 연속형 구간화는 하지 않는다. 로짓 선형성은 Box-Tidwell 로 확인한다 (설계 §3-10).")

⑤ 모델링용 전처리에서 log1p 변환할 변수

  - prior_view_cnt           왜도  5.901  (담당 A · 이슬기)
  - prior_uniq_product_cnt   왜도  6.044  (담당 A · 이슬기)
  - cart_product_view_cnt    왜도  1.751  (담당 A · 이슬기)
  - price                    왜도  7.210  (담당 B · 배지환)

※ 변환은 log 가 아니라 log1p 로 통일한다 — 미결 ① 결정 (2026-08-31, 임시).
※ 반드시 '층화 분할 후'에 변환한다 (LAB-13 04).
※ 이상치는 처리하지 않는다 (설계 §3-5).
※ 연속형 구간화는 하지 않는다. 로짓 선형성은 Box-Tidwell 로 확인한다 (설계 §3-10).


### 4-1. 미결 ① — 결정 내용을 표에 남긴다

`prior_view_cnt`(log1p 후 1.042)와 `prior_uniq_product_cnt`(1.033)는 **변환 후에도 기준 1.0을 넘는다.**
그럼에도 변환하기로 했다. 사유는 A조 `#09` · B조 `#10`에 있고, 요지는 넷이다.

1. 설계 §3-4의 두 조건(대상 · 취소)을 모두 통과했고, **"변환 후 기준"은 설계에 없던 조항**이다 → 지금 추가하면 §12 위반
2. 잔차 왜도는 0 과다가 아니라 **실제 긴 꼬리**(최대 71회)에서 온다 — 0을 전부 빼면 오히려 1.643으로 악화
3. 로지스틱 회귀의 가정 목록(로짓 선형성·VIF·독립성)에 **X의 대칭성은 없다** (브리핑 §7)
4. 기준 1.0은 상관계수 선택용으로 빌려온 관례선이고, 교재가 *"칼로 자르듯 판단 금지"*라 경고한 종류다 `[LAB-07 7단원 p.9]`

> **⚠️ 임시 결정이다.** 이슬기 확인 후 확정한다.

In [8]:
LIMIT = (
    "prior_view_cnt·prior_uniq_product_cnt 는 log1p 변환 후에도 왜도가 "
    "1.042·1.033 으로 사전 기준(1.0)을 넘는다. 잔차 왜도는 조회 건수의 실제 긴 꼬리"
    "(최대 71회, 10회 이상 1.42%)에서 오며 단조변환으로 제거할 수 없다. "
    "로짓 선형성은 ⑤의 Box-Tidwell 로 별도 확인한다."
)

print("보고서 Ⅵ-3 한계 항목에 넣을 문장")
print("-" * 70)
print(LIMIT)

보고서 Ⅵ-3 한계 항목에 넣을 문장
----------------------------------------------------------------------
prior_view_cnt·prior_uniq_product_cnt 는 log1p 변환 후에도 왜도가 1.042·1.033 으로 사전 기준(1.0)을 넘는다. 잔차 왜도는 조회 건수의 실제 긴 꼬리(최대 71회, 10회 이상 1.42%)에서 오며 단조변환으로 제거할 수 없다. 로짓 선형성은 ⑤의 Box-Tidwell 로 별도 확인한다.


---

## #05. 저장

In [9]:
U_PATH = OUT + r"\09_단변량_통합.csv"
U.to_csv(U_PATH, encoding="utf-8-sig")

print("저장 완료 :", U_PATH)
print(f"  {U.shape[0]}행 × {U.shape[1]}열")
print()
print("이 파일이 보고서 Ⅱ-1 과 ⑤ 전처리의 입력이 된다.")

저장 완료 : .\output\09_단변량_통합.csv
  9행 × 11열

이 파일이 보고서 Ⅱ-1 과 ⑤ 전처리의 입력이 된다.


---

## #06. 다음

| # | 할 일 | 누가 |
|---|---|---|
| 4 | **교차 검수** — A조는 배지환, B조는 이슬기 | 서로 |
| 5 | 설계 `§6 조정 기록`에 미결 ① 결정 기록 · `§4-4`에 수치 표기 규칙 추가 | 아무나 |
| 5 | 브리핑 `§1`·`§10` 갱신 (단변량 종료) | 아무나 |
| 6 | **`10` 이변량 진입** | 둘이 함께 |

### 교차 검수에서 볼 것 — 하나면 충분하다

> **판정문에 셀 번호가 없는 수치가 있는가.**

있으면 그 자리가 문제다. 근거를 셀에서 되짚을 수 없는 수치는 쓰지 않는다는 규칙을
A조·B조 양쪽에서 한 번씩 어겼고, 두 번 다 이 방법으로 잡았다.

### `10` 이변량에서 할 일 (미리보기)

```
STEP 3 이변량   연속형 4 → test_independent(wide) + anova_oneway(np2)
                명목형 4 → chi2_independence(V) + 표준화 잔차(brand_grp)
STEP 4 다변량   multi_correlation → correlation_summary → compute_vif
                ↑ 여기서 조회 3종 중 무엇을 남길지 정한다
STEP 5 스코어카드  채택 / 보류 / 제외
```

> **아직 Y를 보지 않았다.** 09번 전체에서 Y와의 관계를 한 번도 들여다보지 않았고,
> 그 덕분에 판정 기준이 결과에 오염되지 않았다. `10`번에서 처음 본다.